# 04 · Evaluation

Scores the best checkpoint on a split it never trained on and writes every chart and table to
`reports/` on Drive.

Produced here: accuracy, precision, recall, F1 (macro and weighted), a confusion matrix, per-class
metrics, one-vs-rest ROC with macro AUC, a false-positive and false-negative breakdown, and a
confidence-threshold sweep for the on-device reject gate.

Which split is used comes from `evaluation.split` (default `test`).

In [ ]:
#@title Setup — mount Drive, locate the project, install what is missing
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MahmoudMabrok/SaloAleh.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def find_project_root() -> Path:
    """The folder that holds src/ and configs/ — cloned if it is not here yet."""
    candidates = [Path(p) for p in [
        os.environ.get("DHIKR_PROJECT_ROOT", ""),
        "/content/DhikrSpeech",
        "/content/SaloAleh/DhikrSpeech",
        "/content/drive/MyDrive/DhikrSpeech",
    ] if p]
    candidates += [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "config.py").is_file() and (candidate / "configs" / "config.yaml").is_file():
            return candidate.resolve()
    if IN_COLAB:
        target = Path("/content/SaloAleh")
        if not target.exists():
            print("cloning", REPO_URL)
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
        return (target / "DhikrSpeech").resolve()
    raise FileNotFoundError(
        "DhikrSpeech project not found. Set DHIKR_PROJECT_ROOT, or copy the "
        "DhikrSpeech folder to /content or to your Drive."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for module_name, package in [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("yaml", "PyYAML"),
    ("sklearn", "scikit-learn"),
    ("soxr", "soxr"),
]:
    if importlib.util.find_spec(module_name) is None:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
config.paths.ensure_dirs()

print("project root :", PROJECT_ROOT)
print("config       :", CONFIG_PATH)
print()
print(config.summary())


## 1 · Load the model and the evaluation split

In [ ]:
import pandas as pd

from src.dataset import class_names_from_manifest, filter_split, load_manifest, make_tf_dataset
from src.features import FeatureStats, LogMelExtractor
from src.trainer import load_trained_model

RUN_NAME = config.model.name

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

split_name = config.evaluation.split
eval_records = filter_split(records, split_name)
if not eval_records:
    fallback = "val"
    print("split %r is empty — falling back to %r" % (split_name, fallback))
    split_name, eval_records = fallback, filter_split(records, fallback)
if not eval_records:
    raise ValueError("no clips to evaluate — re-run 02_preprocessing")

checkpoint = paths.checkpoints_path / RUN_NAME / "best_model.keras"
model = load_trained_model(checkpoint)

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None
extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)

# No shuffling and no augmentation: predictions line up with eval_records.
eval_dataset = make_tf_dataset(
    eval_records,
    config,
    extractor,
    training=False,
    batch_size=config.evaluation.batch_size,
    shuffle=False,
)

print("checkpoint :", checkpoint)
print("split      : %s (%d clips)" % (split_name, len(eval_records)))
print("classes    :", len(class_names))


## 2 · Predict and score

In [ ]:
from src.metrics import evaluate_model

result = evaluate_model(
    model,
    eval_dataset,
    class_names,
    paths=[record.path for record in eval_records],
    confidence_threshold=config.evaluation.confidence_threshold,
)
print(result.summary())

macro, weighted = result.averaged("macro"), result.averaged("weighted")
display(pd.DataFrame([
    {"average": "macro", **{key: round(value, 4) for key, value in macro.items()}},
    {"average": "weighted", **{key: round(value, 4) for key, value in weighted.items()}},
]))


## 3 · Per-class metrics

Sorted worst first — the classes at the top are the ones that need more recordings.

In [ ]:
from src import visualization as viz

per_class = result.per_class()
table = result.to_dataframe().sort_values("f1")
display(table)

figure = viz.plot_per_class_metrics(per_class)
viz.save_figure(figure, paths.reports_path / "04_per_class_metrics.png")

weak = table[table["f1"] < 0.8]
if len(weak):
    print("classes below 0.80 F1:", ", ".join(weak["label"].tolist()))
else:
    print("every class is at or above 0.80 F1")


## 4 · Confusion matrix

Rows are the true class, columns the prediction. Bright cells off the diagonal are the phrase pairs
the model mixes up — usually phrases that share a leading word.

In [ ]:
matrix = result.confusion_matrix

figure = viz.plot_confusion_matrix(matrix, class_names, normalize=True,
                                   title="confusion matrix (row-normalised)")
viz.save_figure(figure, paths.reports_path / "04_confusion_matrix.png")

figure = viz.plot_confusion_matrix(matrix, class_names, normalize=False,
                                   title="confusion matrix (counts)")
viz.save_figure(figure, paths.reports_path / "04_confusion_matrix_counts.png")

confusions = result.top_confusions(config.evaluation.top_k_confusions)
if confusions:
    display(pd.DataFrame(confusions, columns=["true", "predicted", "count"]))
else:
    print("no off-diagonal errors")


## 5 · ROC

One-vs-rest per class. A class with no examples in this split is skipped, since its AUC is
undefined.

In [ ]:
if config.evaluation.roc:
    curves = result.roc_curves()
    macro_auc = curves.get("__macro__", {}).get("auc")
    print("macro AUC:", round(macro_auc, 4) if macro_auc is not None else "n/a")

    figure = viz.plot_roc_curves(curves)
    viz.save_figure(figure, paths.reports_path / "04_roc_curves.png")

    display(pd.DataFrame(
        [
            {"class": label, "auc": round(float(payload["auc"]), 4)}
            for label, payload in curves.items()
            if label != "__macro__"
        ]
    ).sort_values("auc"))
else:
    print("evaluation.roc is false — skipped")


## 6 · False positives and false negatives

* **False positive** — another phrase was predicted as this class. On device this is a phantom count.
* **False negative** — this class was said but predicted as something else. On device this is a missed count.

Listed for the classes with the most errors, with the file path so the clip can be listened to.

In [ ]:
from dataclasses import asdict

worst = sorted(per_class, key=lambda item: item.false_positives + item.false_negatives, reverse=True)
limit = config.evaluation.error_examples

for metrics in worst[:5]:
    if metrics.false_positives == 0 and metrics.false_negatives == 0:
        continue
    print("=" * 78)
    print("%s — %d false positive(s), %d false negative(s), support %d"
          % (metrics.label, metrics.false_positives, metrics.false_negatives, metrics.support))

    false_positives = result.false_positives(metrics.label, limit)
    if false_positives:
        print("\nfalse positives (predicted %s, actually something else):" % metrics.label)
        display(pd.DataFrame([asdict(case) for case in false_positives]))

    false_negatives = result.false_negatives(metrics.label, limit)
    if false_negatives:
        print("\nfalse negatives (%s said, predicted otherwise):" % metrics.label)
        display(pd.DataFrame([asdict(case) for case in false_negatives]))

if not any(item.false_positives or item.false_negatives for item in per_class):
    print("no errors on this split")


## 7 · Listen to the errors

The fastest way to tell a model problem from a data problem: if the clip sounds like the predicted
phrase, the label is wrong, not the model.

In [ ]:
from IPython.display import Audio, display

from src.audio import read_wav

errors = result.all_errors(limit=8)
if not errors:
    print("no misclassified clips to play")
for case in errors:
    print("true %-10s predicted %-10s confidence %.3f  %s"
          % (case.true_label, case.predicted_label, case.confidence, case.path))
    try:
        clip, _ = read_wav(paths.processed_path / case.path)
        display(Audio(clip, rate=config.audio.sample_rate))
    except Exception as error:
        print("  could not load:", error)


## 8 · Confidence threshold

On device the model runs continuously, so a prediction below a threshold should be discarded rather
than counted. This sweep picks that threshold: raise it until the error rate is acceptable, then
check how many correct detections it costs.

In [ ]:
import numpy as np

correct = result.y_true == result.y_pred
figure = viz.plot_confidence_distribution(
    result.confidence, correct, threshold=config.evaluation.confidence_threshold
)
viz.save_figure(figure, paths.reports_path / "04_confidence_distribution.png")

sweep = pd.DataFrame([
    result.rejection_stats(threshold) for threshold in np.arange(0.0, 1.0, 0.05)
])
sweep["accuracy_on_accepted"] = sweep["accuracy_on_accepted"].round(4)
sweep["accept_rate"] = sweep["accept_rate"].round(4)
display(sweep)

print()
print("current threshold (evaluation.confidence_threshold = %.2f):"
      % config.evaluation.confidence_threshold)
print(result.rejection_stats())


## 9 · Save the report

Written to `reports/`:

* `evaluation.json` — all metrics
* `evaluation_per_class.csv`
* `evaluation_errors.csv` — every misclassified clip
* `evaluation_confusion_matrix.csv`
* `04_*.png` — every chart above

In [ ]:
written = result.save(paths.reports_path)
for kind, path in written.items():
    print("%-18s %s" % (kind, path))

print()
print("charts:")
for path in sorted(paths.reports_path.glob("04_*.png")):
    print("  ", path)

print()
print("accuracy %.4f on the %s split — continue with 05_export.ipynb"
      % (result.accuracy, split_name))
